# Controlling MODFLOW 6 with the API — F. Sequential reactions (denitrification)

This is part F of the MODFLOW 6 API series — see [**A. Basic usage**](modflow-api-A.ipynb) for the API lifecycle and [**C. Adjusting recharge with a callback**](modflow-api-C.ipynb) for the callback mechanism this example builds on.

Use the API to add *chemistry* that MODFLOW 6 does not solve on its own. A single simulation holds one groundwater-flow (GWF) model and **three** groundwater-transport (GWT) models — one per nitrogen species — wired to flow with GWF-GWT exchanges and sharing a single transport solution. At each transport time step a `modflowapi` callback applies a first-order sequential denitrification chain

$$\mathrm{NO_3} \;\xrightarrow{\;k_1\;}\; \mathrm{NO_2} \;\xrightarrow{\;k_2\;}\; \mathrm{N_2}$$

by reading each species' concentration from MODFLOW's memory and writing the reaction mass rates back. A lake is the nitrate source; nitrite and dinitrogen start at zero everywhere, so their plumes are *purely reaction products*.

The base flow-and-transport model is the MODFLOW 6 example `ex-gwt-prudic2004t2` (Prudic and others, 2004), which couples SFR (streams), LAK (a lake), and MVR (a mover) to advanced-package transport (SFT/LKT/MVT).

Run the imports and library-location cells below first.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the supporting plotting and data libraries, then create a directory for the figures saved by this notebook.

In [ ]:
%matplotlib inline

import math
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
from flopy.plot.styles import styles
from matplotlib.colors import ListedColormap, LogNorm
from matplotlib.ticker import FuncFormatter, StrMethodFormatter
from modflowapi import Callbacks, run_simulation
from notebook_helpers import find_mf6_libraries

In [ ]:
fig_path = pl.Path(".figures")
fig_path.mkdir(exist_ok=True)

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the `mf6` command-line executable. The `find_mf6_libraries` helper in `notebook_helpers.py` locates the library (`.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the `mf6` executable inside the active pixi environment; confirm that both exist.

In [ ]:
# libmf6 (the shared library the API drives) and the mf6 executable,
# located in the active pixi environment
lib_name, mf6_exe = find_mf6_libraries()

In [ ]:
lib_name.is_file(), mf6_exe.is_file()

## The denitrification problem

Set the model name and workspace, then the grid, flow, and transport parameters (all taken from `ex-gwt-prudic2004t2`). The flow model is steady state; transport runs for 25 years, split into `nstp_transport` time steps, and the API callback fires once per step.

Pick the **reaction-coupling method** here with `reaction_method`:

- `"src_rhs"` — the callback computes the reaction mass rates and writes them into a Mass Source Loading (SRC) package for each species. SRC adds the terms to the solution right-hand side *and* books them in the GWT budget, so every per-model mass budget closes.
- `"operator_split"` — the callback overwrites each concentration array in place at the end of the time step (operator splitting). Simpler, but the reacted mass shows up as a per-model budget discrepancy because it is not a recognized package term.

In [ ]:
sim_name = "nitrate-nitrite"
base_example = "ex-gwt-prudic2004t2"

# API run workspace (under the git-ignored models/ directory) and the base-model
# data directory (the grid/boundary/stream files shipped alongside this notebook)
workspace = pl.Path(f"models/{sim_name}_api_F")
data_path = pl.Path("../data") / base_example

# reaction-coupling method: "src_rhs" (clean budget) or "operator_split"
reaction_method = "src_rhs"

# --- flow / transport parameters (from ex-gwt-prudic2004t2) ---
length_units = "feet"
time_units = "days"

hk = 250.0
vk = 125.0
porosity = 0.30
recharge = 4.79e-3
lakebed_leakance = 1.0
streambed_k = 100.0
streambed_thick = 1.0
stream_width = 5.0
manning = 0.03
alpha_l = 20.0
alpha_th = 2.0
alpha_tv = 0.2

nlay = 8
nrow = 36
ncol = 23
delr = 405.665
delc = 403.717
delv = 15.0
top = 100.0
total_time = 9131.0
nstp_transport = 300

### Reaction parameters

The chain is two first-order reactions, `NO3 -> NO2 -> N2`, each set by a half-life. The mass yields (`y1`, `y2`) come from molar stoichiometry, so the mass that leaves a parent species is added to its daughter exactly (times the molar-mass ratio).

The lake nitrate concentration follows a *source-reduction scenario*: held at `no3_source_conc` until year 10, then reduced linearly to `no3_source_conc_final` (below the drinking-water limit) by year 15, and held there afterward. Only the larger lake supplies nitrate; the smaller lake is kept at zero.

In [ ]:
SUSTAINED_SOURCE = True  # True: lake held at a (time-varying) NO3; False: slug
no3_source_conc = 50.0  # initial lake nitrate concentration (mg/L)

# source-reduction time series: constant until year 10, reduced to just below the
# nitrate MCL by year 15, then held there
no3_source_conc_final = 10.0
source_reduce_start = 10.0 * 365.0  # days (10 years)
source_reduce_end = 15.0 * 365.0  # days (15 years)
source_ts = [
    (0.0, no3_source_conc),
    (source_reduce_start, no3_source_conc),
    (source_reduce_end, no3_source_conc_final),
    (total_time, no3_source_conc_final),
]

half_life_no3 = 730.0  # NO3 first-order half-life (days)
half_life_no2 = 365.0  # NO2 first-order half-life (days)
k1 = math.log(2.0) / half_life_no3  # NO3 -> NO2 rate (1/day)
k2 = math.log(2.0) / half_life_no2  # NO2 -> N2  rate (1/day)

# species molar masses (g/mol) and the mass yields from molar stoichiometry
M_NO3 = 62.004
M_NO2 = 46.005
M_N2 = 28.014
y1 = M_NO2 / M_NO3  # 1 mol NO3 -> 1 mol NO2  (~0.742 mass)
y2 = 0.5 * M_N2 / M_NO2  # 1 mol NO2 -> 0.5 mol N2 (~0.305 mass)

SPECIES = ["no3", "no2", "n2"]

# map display: mask concentrations below MASK_BELOW (log color scale) and draw the
# drinking-water maximum contaminant level (MCL) contour where one exists. EPA MCLs
# are 10 mg/L as N for nitrate (= 44.3 mg/L as NO3) and 1 mg/L as N for nitrite
# (= 3.28 mg/L as NO2); dinitrogen has none.
MASK_BELOW = 0.01
MCL = {"no3": 44.3, "no2": 3.28}

### Read the base-model data

The grid, boundary, and stream data for `ex-gwt-prudic2004t2` are stored in `../data/ex-gwt-prudic2004t2` next to this notebook (originally from the MODFLOW 6 examples repository). Define helpers that read those files and parse the grid arrays and the SFR stream geometry.

In [ ]:
def retrieve(fname):
    """Return the path to a base-model data file in the local data directory."""
    dest = data_path / fname
    if not dest.is_file():
        raise FileNotFoundError(
            f"{dest} not found - expected the {base_example} data alongside "
            "this notebook (see ../data)."
        )
    return dest


def load_grid_data():
    bot0 = np.loadtxt(retrieve("bot1.dat"))
    botm = [bot0] + [bot0 - (delv * k) for k in range(1, nlay)]
    idomain0 = np.loadtxt(retrieve("idomain1.dat"), dtype=int)
    idomain = nlay * [idomain0]
    lakibd = np.loadtxt(retrieve("lakibd.dat"), dtype=int)
    return botm, idomain, lakibd


def get_stream_data():
    fpath = retrieve("stream.csv")
    dt = 5 * [int] + [float]
    streamdata = np.genfromtxt(fpath, names=True, delimiter=",", dtype=dt)
    connectiondata = [[ireach] for ireach in range(streamdata.shape[0])]
    isegold = -1
    distance_along_segment = []
    distance = 0
    for ireach, row in enumerate(streamdata):
        iseg = row["seg"] - 1
        if iseg == isegold:
            connectiondata[ireach].append(ireach - 1)
            connectiondata[ireach - 1].append(-ireach)
            distance += (
                streamdata["length"][ireach - 1] * 0.5
                + streamdata["length"][ireach] * 0.5
            )
        else:
            distance = 0.5 * streamdata["length"][ireach]
        isegold = iseg
        distance_along_segment.append(distance)

    connectiondata[17].append(-31)
    connectiondata[31].append(17)
    connectiondata[30].append(-31)
    connectiondata[31].append(30)

    segment_lengths = []
    for iseg in [1, 2, 3, 4]:
        idx = np.where(streamdata["seg"] == iseg)
        segment_lengths.append(streamdata["length"][idx].sum())

    emaxmin = [(49, 45), (44.5, 34), (41.5, 34.0), (34.0, 27.2)]
    segment_gradients = [
        (emax - emin) / segment_lengths[iseg]
        for iseg, (emax, emin) in enumerate(emaxmin)
    ]

    ustrf = 1.0
    ndv = 0
    packagedata = []
    for ireach, row in enumerate(streamdata):
        k, i, j = row["layer"] - 1, row["row"] - 1, row["col"] - 1
        length = row["length"]
        iseg = row["seg"] - 1
        rgrd = segment_gradients[iseg]
        emax, emin = emaxmin[iseg]
        rtp = distance_along_segment[ireach] / segment_lengths[iseg] * (emax - emin)
        rtp = emax - rtp
        rec = (
            ireach,
            (k, i, j),
            length,
            stream_width,
            rgrd,
            rtp,
            streambed_thick,
            streambed_k,
            manning,
            len(connectiondata[ireach]) - 1,
            ustrf,
            ndv,
            f"SEG{iseg + 1}",
        )
        packagedata.append(rec)
    return packagedata, connectiondata

### Build the coupled flow + 3× transport simulation

`build_simulation` assembles one `MFSimulation` holding the GWF flow model and one GWT model per species. Each GWT is registered with the shared transport IMS solution (one model at a time — passing a list registers only the first) and wired to flow with a GWF-GWT exchange, so the callback sees all three species in the same solution. When `reaction_method == "src_rhs"`, every species also gets a Mass Source Loading (SRC) package with one entry per active cell that the callback overwrites each step.

In [ ]:
def build_simulation():
    botm, idomain, lakibd = load_grid_data()

    sim = flopy.mf6.MFSimulation(
        sim_name=sim_name, sim_ws=workspace, exe_name=str(mf6_exe)
    )
    flopy.mf6.ModflowTdis(
        sim,
        nper=1,
        perioddata=[(total_time, nstp_transport, 1.0)],
        time_units=time_units,
    )

    gwf = _build_gwf(sim, botm, idomain, lakibd)
    gwts = [
        _build_gwt(sim, name, botm, lakibd, is_source=(name == "no3"))
        for name in SPECIES
    ]

    imsgwf = flopy.mf6.ModflowIms(
        sim,
        print_option="summary",
        outer_maximum=1000,
        inner_maximum=100,
        outer_dvclose=1.0e-3,
        inner_dvclose=1.0e-4,
        rcloserecord=[1.0e-3, "strict"],
        relaxation_factor=0.99,
        filename="flow.ims",
    )
    imsgwt = flopy.mf6.ModflowIms(
        sim,
        print_option="summary",
        outer_maximum=100,
        inner_maximum=100,
        outer_dvclose=1.0e-4,
        inner_dvclose=1.0e-5,
        rcloserecord=[1.0e-4, "strict"],
        under_relaxation="DBD",
        under_relaxation_theta=0.7,
        linear_acceleration="bicgstab",
        relaxation_factor=0.97,
        filename="trans.ims",
    )

    # register each model with its solution one at a time (a multi-model list to
    # register_ims_package silently drops all but the first model)
    sim.register_ims_package(imsgwf, gwf.name)
    for gwt in gwts:
        sim.register_ims_package(imsgwt, gwt.name)
        flopy.mf6.ModflowGwfgwt(
            sim,
            exgtype="GWF6-GWT6",
            exgmnamea=gwf.name,
            exgmnameb=gwt.name,
            filename=f"{gwf.name}_{gwt.name}.gwfgwt",
        )

    return sim


def _build_gwf(sim, botm, idomain, lakibd):
    name = "flow"
    gwf = flopy.mf6.ModflowGwf(sim, modelname=name, save_flows=True)
    dis = flopy.mf6.ModflowGwfdis(
        gwf,
        length_units=length_units,
        nlay=nlay,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        top=top,
        botm=botm,
        idomain=idomain,
    )
    flopy.mf6.ModflowGwfnpf(
        gwf,
        save_specific_discharge=True,
        save_saturation=True,
        icelltype=[1] + 7 * [0],
        k=hk,
        k33=vk,
    )
    flopy.mf6.ModflowGwfic(gwf, strt=50.0)
    flopy.mf6.ModflowGwfoc(
        gwf,
        head_filerecord=f"{name}.hds",
        budget_filerecord=f"{name}.bud",
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    )
    flopy.mf6.ModflowGwfrcha(gwf, recharge={0: recharge}, pname="RCH-1")

    chdlist = []
    fpath = retrieve("chd.dat")
    for line in open(fpath).readlines():
        ll = line.strip().split()
        if len(ll) == 4:
            k, i, j, hd = ll
            chdlist.append([(int(k) - 1, int(i) - 1, int(j) - 1), float(hd)])
    flopy.mf6.ModflowGwfchd(gwf, stress_period_data=chdlist, pname="CHD-1")

    idomain = dis.idomain.array
    lake_map = np.ones((nlay, nrow, ncol), dtype=np.int32) * -1
    lake_map[0, :, :] = lakibd[:, :] - 1
    idomain, lakepakdata_dict, lakeconnectiondata = flopy.mf6.utils.get_lak_connections(
        gwf.modelgrid, lake_map, idomain=idomain, bedleak=lakebed_leakance
    )
    gwf.dis.idomain.set_data(idomain[0], layer=0, multiplier=[1])
    lakpackagedata = [
        [0, 44.0, lakepakdata_dict[0], "lake1"],
        [1, 35.2, lakepakdata_dict[1], "lake2"],
    ]
    outlets = [[0, 0, -1, "MANNING", 44.5, 3.36493214532915, 0.03, 0.2187500e-02]]
    flopy.mf6.ModflowGwflak(
        gwf,
        time_conversion=86400.0,
        length_conversion=3.28081,
        print_stage=True,
        print_flows=True,
        stage_filerecord=name + ".lak.bin",
        budget_filerecord=name + ".lak.bud",
        mover=True,
        pname="LAK-1",
        boundnames=True,
        nlakes=len(lakpackagedata),
        noutlets=len(outlets),
        outlets=outlets,
        packagedata=lakpackagedata,
        connectiondata=lakeconnectiondata,
    )

    sfrpackagedata, sfrconnectiondata = get_stream_data()
    sfrperioddata = {0: [[0, "inflow", 86400], [18, "inflow", 8640.0]]}
    flopy.mf6.ModflowGwfsfr(
        gwf,
        print_stage=True,
        print_flows=True,
        stage_filerecord=name + ".sfr.bin",
        budget_filerecord=name + ".sfr.bud",
        mover=True,
        pname="SFR-1",
        time_conversion=86400.0,
        length_conversion=3.28081,
        boundnames=True,
        nreaches=len(sfrconnectiondata),
        packagedata=sfrpackagedata,
        connectiondata=sfrconnectiondata,
        perioddata=sfrperioddata,
    )
    flopy.mf6.ModflowGwfmvr(
        gwf,
        maxmvr=2,
        print_flows=True,
        budget_filerecord=name + ".mvr.bud",
        maxpackages=2,
        packages=[["SFR-1"], ["LAK-1"]],
        perioddata=[
            ["SFR-1", 5, "LAK-1", 0, "FACTOR", 1.0],
            ["LAK-1", 0, "SFR-1", 6, "FACTOR", 1.0],
        ],
    )
    return gwf


def _build_gwt(sim, name, botm, lakibd, is_source):
    """Build one GWT species model. is_source -> the lake supplies this species."""
    gwt = flopy.mf6.ModflowGwt(sim, modelname=name, save_flows=True)
    idomain = sim.get_model("flow").dis.idomain.array

    flopy.mf6.ModflowGwtdis(
        gwt,
        length_units=length_units,
        nlay=nlay,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        top=top,
        botm=botm,
        idomain=idomain,
    )
    flopy.mf6.ModflowGwtic(gwt, strt=0.0)
    flopy.mf6.ModflowGwtmst(gwt, porosity=porosity)
    flopy.mf6.ModflowGwtadv(gwt, scheme="TVD")
    flopy.mf6.ModflowGwtdsp(gwt, alh=alpha_l, ath1=alpha_th, ath2=alpha_tv)
    flopy.mf6.ModflowGwtssm(gwt, sources=[[]])

    # SRC package for the reaction terms (used only by the "src_rhs" method). One
    # entry per active cell; the API overwrites the loading rates each time step.
    if reaction_method == "src_rhs":
        active_cells = _active_cellids(idomain)
        srcdata = [[cid, 0.0] for cid in active_cells]
        flopy.mf6.ModflowGwtsrc(
            gwt,
            stress_period_data=srcdata,
            maxbound=len(srcdata),
            save_flows=True,
            pname="SRC-1",
        )

    # LKT: lake transport (the nitrate source lives here)
    lake_strt = no3_source_conc if (is_source and not SUSTAINED_SOURCE) else 0.0
    lktpackagedata = [
        (0, lake_strt, 99.0, 999.0, "mylake1"),
        (1, lake_strt, 99.0, 999.0, "mylake2"),
    ]
    if is_source and SUSTAINED_SOURCE:
        # lake1 nitrate held at a time-varying concentration (a time series);
        # lake2 held at zero nitrate for the whole simulation
        lktperioddata = [
            (0, "STATUS", "CONSTANT"),
            (0, "CONCENTRATION", "lakeconc"),
            (1, "STATUS", "CONSTANT"),
            (1, "CONCENTRATION", 0.0),
        ]
    else:
        lktperioddata = [(0, "STATUS", "ACTIVE"), (1, "STATUS", "ACTIVE")]
    lkt = flopy.mf6.modflow.ModflowGwtlkt(
        gwt,
        boundnames=True,
        save_flows=True,
        print_concentration=True,
        concentration_filerecord=name + ".lkt.bin",
        budget_filerecord=name + ".lkt.bud",
        packagedata=lktpackagedata,
        lakeperioddata=lktperioddata,
        pname="LAK-1",
        auxiliary=["aux1", "aux2"],
    )
    if is_source and SUSTAINED_SOURCE:
        lkt.ts.initialize(
            filename=name + ".lkt.ts",
            timeseries=source_ts,
            time_series_namerecord=["lakeconc"],
            interpolation_methodrecord=["linear"],
        )

    # SFT: stream transport
    sfrpackagedata, sfrconnectiondata = get_stream_data()
    nreach = len(sfrconnectiondata)
    sftpackagedata = [
        (irno, 0.0, 99.0, 999.0, f"myreach{irno + 1}") for irno in range(nreach)
    ]
    flopy.mf6.modflow.ModflowGwtsft(
        gwt,
        boundnames=True,
        save_flows=True,
        print_concentration=True,
        concentration_filerecord=name + ".sft.bin",
        budget_filerecord=name + ".sft.bud",
        packagedata=sftpackagedata,
        reachperioddata=[(0, "STATUS", "ACTIVE")],
        pname="SFR-1",
        auxiliary=["aux1", "aux2"],
    )
    flopy.mf6.modflow.ModflowGwtmvt(gwt, print_flows=True)

    flopy.mf6.ModflowGwtoc(
        gwt,
        budget_filerecord=f"{name}.bud",
        concentration_filerecord=f"{name}.ucn",
        saverecord=[("CONCENTRATION", "ALL"), ("BUDGET", "ALL")],
        printrecord=[("CONCENTRATION", "LAST"), ("BUDGET", "ALL")],
    )
    return gwt


def _active_cellids(idomain):
    """(k, i, j) for active cells (idomain > 0) in MF6 reduced-node order."""
    cells = []
    for k in range(nlay):
        for i in range(nrow):
            for j in range(ncol):
                if idomain[k, i, j] > 0:
                    cells.append((k, i, j))
    return cells

Build the simulation and write the input files. The `src_rhs` method adds a SRC package (one entry per active cell) to each transport model; the `operator_split` method does not.

In [ ]:
sim = build_simulation()
sim.write_simulation(silent=True)
print(f"reaction method: {reaction_method}")
print("wrote simulation to", workspace)

## Apply the reaction with an API callback

The reaction chain is not a MODFLOW package — the API supplies it. Two callbacks are defined below; `reaction_method` (set above) selects which one runs.

- **`SrcRhsCallback`** (`src_rhs`) fires at the *start* of each time step. It reads the current aquifer concentrations, computes the first-order mass rates `r1 = k1·[NO3]·Vp` and `r2 = k2·[NO2]·Vp` (with `Vp` the saturated pore volume of each cell), and writes them into each species' SRC loading array (`SMASSRATE`). MODFLOW then adds those terms to the solution and books them in the budget, so mass transfer between species is exact and every per-model budget closes.
- **`OperatorSplitCallback`** (`operator_split`) fires at the *end* of each time step and decays the concentration arrays in place: `NO3` loses `1 - e^{-k1·dt}` of its mass, `NO2` gains the nitrate yield and loses its own, and `N2` gains the nitrite yield. Simpler, but the reacted mass is invisible to the budget.

The GWT solution vector `X` is laid out as `[aquifer cells | LKT lakes | SFT reaches]`; the reaction is applied only to the aquifer cells (the first `naq` entries).

First, two helpers compute the per-cell saturated pore volume the SRC method needs and the total dissolved mass of each species over time.

In [ ]:
def saturated_pore_volume_reduced(sim):
    """Pore volume (porosity * cell volume) per active node, in MF6 reduced order.
    Saturation is applied at run time from the live NPF SAT array."""
    botm, _, _ = load_grid_data()
    idomain = sim.get_model("flow").dis.idomain.array
    thickness = np.empty((nlay, nrow, ncol))
    thickness[0] = top - botm[0]
    for k in range(1, nlay):
        thickness[k] = botm[k - 1] - botm[k]
    vol = (delr * delc) * thickness  # cell volume (ft^3)
    active = idomain.ravel() > 0
    return porosity * vol.ravel()[active]


def total_mass_in_storage(sim):
    """Dissolved mass (kg) of each species in the aquifer through time. Flow is
    steady, so the saturated thickness (from the water table) is constant."""
    botm, _, _ = load_grid_data()
    gwf = sim.get_model("flow")
    idomain = gwf.dis.idomain.array
    head = gwf.output.head().get_data()  # steady state
    thickness = np.empty((nlay, nrow, ncol))
    thickness[0] = top - botm[0]
    for k in range(1, nlay):
        thickness[k] = botm[k - 1] - botm[k]
    sat_thick = thickness.astype(float).copy()
    sat_thick[0] = np.clip(head[0] - botm[0], 0.0, thickness[0])  # water table
    water_vol = porosity * (delr * delc) * sat_thick  # ft^3 of water / cell
    active = idomain > 0
    L_per_ft3 = 28.316846592
    times, mass = None, {}
    for s in SPECIES:
        cobj = sim.get_model(s).output.concentration()
        times = np.array(cobj.times) / 365.0
        calldata = cobj.get_alldata()  # (ntime, nlay, nrow, ncol)
        cmasked = np.where(active[None] & (calldata < 1.0e29), calldata, 0.0)
        # mg/L * ft^3 * (L/ft^3) = mg; / 1e6 -> kg
        mass[s] = (cmasked * water_vol[None]).sum(axis=(1, 2, 3)) * L_per_ft3 / 1.0e6
    return times, mass

In [ ]:
class OperatorSplitCallback:
    """Apply the NO3->NO2->N2 chain by overwriting the state arrays X at the end
    of each transport time step (operator splitting)."""

    def __init__(self):
        self.history = []
        self.elapsed = 0.0
        self._ptrs = None

    def __call__(self, sim, step):
        if step != Callbacks.timestep_end:
            return
        models = {m.name.lower(): m for m in sim.models}
        if not all(s in models for s in SPECIES):
            return
        if self._ptrs is None:
            mf6 = models["no3"].mf6
            self._ptrs = {
                s: mf6.get_value_ptr(mf6.get_var_address("X", models[s].name))
                for s in SPECIES
            }
            # X is [aquifer cells | LKT lakes | SFT reaches]; react the aquifer only
            self.naq = mf6.get_value(mf6.get_var_address("X", "FLOW")).shape[0]
        c, naq = self._ptrs, self.naq
        dt = sim.delt
        d1 = np.maximum(c["no3"][:naq], 0.0) * (1.0 - math.exp(-k1 * dt))  # NO3 lost
        d2 = np.maximum(c["no2"][:naq], 0.0) * (1.0 - math.exp(-k2 * dt))  # NO2 lost
        # in-place updates write straight into MF6 memory (do not rebind)
        c["no3"][:naq] -= d1
        c["no2"][:naq] += y1 * d1 - d2
        c["n2"][:naq] += y2 * d2
        self._record(dt, c, naq)

    def _record(self, dt, c, naq):
        self.elapsed += dt
        self.history.append((self.elapsed, *(float(c[s][:naq].sum()) for s in SPECIES)))


class SrcRhsCallback:
    """Compute the reaction mass rates from the current concentrations and write
    them into each species' SRC package at the start of each time step. The SRC
    terms enter the solution RHS and are booked in the GWT budget, so every
    per-model mass budget closes."""

    def __init__(self, cell_pore_volume):
        self.pv = cell_pore_volume  # porosity*volume per active node (reduced)
        self.history = []
        self.elapsed = 0.0
        self._x = None  # live concentration views
        self._smr = None  # live SRC SMASSRATE views
        self._red = None  # SRC entry index -> reduced node
        self._sat_addr = None

    def __call__(self, sim, step):
        if step != Callbacks.timestep_start:
            return
        models = {m.name.lower(): m for m in sim.models}
        if not all(s in models for s in SPECIES):
            return
        mf6 = models["no3"].mf6
        if self._x is None:
            self._x = {
                s: mf6.get_value_ptr(mf6.get_var_address("X", models[s].name))
                for s in SPECIES
            }
            # SMASSRATE (not BOUND) is the loading array MF6 uses to formulate the
            # SRC contribution to the solution
            self._smr = {
                s: mf6.get_value_ptr(
                    mf6.get_var_address("SMASSRATE", models[s].name, "SRC-1")
                )
                for s in SPECIES
            }
            # SRC entries were created in reduced-node order, but map explicitly
            # via NODELIST to be safe
            self._red = {
                s: mf6.get_value(
                    mf6.get_var_address("NODELIST", models[s].name, "SRC-1")
                ).astype(int)
                - 1
                for s in SPECIES
            }
            self._sat_addr = mf6.get_var_address("SAT", "FLOW", "NPF")
            # aquifer node count (X also carries LKT/SFT unknowns past this)
            self.naq = self.pv.shape[0]
            assert self._x["no3"][: self.naq].shape == self.pv.shape

        naq = self.naq
        sat = mf6.get_value(self._sat_addr)
        satpv = self.pv * sat  # saturated pore volume
        c = {s: self._x[s][:naq] for s in SPECIES}  # aquifer concentrations
        # first-order reaction mass rates (mass / time) per active node
        r1 = k1 * np.maximum(c["no3"], 0.0) * satpv  # NO3 -> NO2
        r2 = k2 * np.maximum(c["no2"], 0.0) * satpv  # NO2 -> N2
        rate = {
            "no3": -r1,  # nitrate consumed
            "no2": y1 * r1 - r2,  # nitrite produced then consumed
            "n2": y2 * r2,  # dinitrogen produced
        }
        for s in SPECIES:
            self._smr[s][:] = rate[s][self._red[s]]
        self._record(sim.delt, c)

    def _record(self, dt, c):
        self.elapsed += dt
        self.history.append((self.elapsed, *(float(c[s].sum()) for s in SPECIES)))

### Run the model through the API

Create the callback that matches `reaction_method` and hand it to `run_simulation`, which drives the compiled library through the whole simulation, calling back at every time step so the reaction is applied as the transport solution advances.

In [ ]:
if reaction_method == "src_rhs":
    callback = SrcRhsCallback(saturated_pore_volume_reduced(sim))
else:
    callback = OperatorSplitCallback()

run_simulation(lib_name, workspace, callback, verbose=False)

### Check the mass budgets

Read the final-time-step percent discrepancy from each species' listing file. With `src_rhs` the reaction is a booked SRC term, so every per-model budget should close to a tiny fraction of a percent. With `operator_split` the reacted mass is *not* a package term, so each budget shows a discrepancy roughly equal to the mass converted to or from that species.

In [ ]:
print(f"Per-model mass-balance discrepancy (last time step), method={reaction_method}:")
for s in SPECIES:
    pct = None
    for line in (workspace / f"{s}.lst").read_text().splitlines():
        if "PERCENT DISCREPANCY" in line:
            try:
                pct = float(line.split("=")[-1])
            except ValueError:
                pass
    print(f"  {s}: percent discrepancy = {pct}")

### Concentration maps

Map each species at year 10 (the end of the sustained-source period) across four model layers. Concentrations below `MASK_BELOW` are masked and the color scale is logarithmic; the white contour marks the drinking-water MCL where one exists. The magenta star in panel A is the observation cell used for the breakthrough curves below.

In [ ]:
decades = [0.01, 0.1, 1.0, 10.0, 100.0]
plain = FuncFormatter(lambda v, _: f"{v:g}")

lakibd = load_grid_data()[2]
gwf = sim.get_model("flow")
labels = {
    "no3": "Nitrate (NO$_3$)",
    "no2": "Nitrite (NO$_2$)",
    "n2": "Dinitrogen (N$_2$)",
}
map_layers = [0, 2, 4, 7]

# observation cell (layer 1, row 18, col 18), 0-based
obs = (0, 17, 17)
xc = gwf.modelgrid.xcellcenters[obs[1], obs[2]]
yc = gwf.modelgrid.ycellcenters[obs[1], obs[2]]
lake_cmap = ListedColormap(["#6ca6cd"])
lake_overlay = np.ma.masked_where(lakibd == 0, np.ones((nrow, ncol)))

# maps are shown at 10 years (closest transport output time)
_mt = np.array(sim.get_model("no3").output.concentration().times)
map_totim = float(_mt[np.argmin(np.abs(_mt - 10.0 * 365.0))])
map_year = map_totim / 365.0

for s in SPECIES:
    gwt = sim.get_model(s)
    conc = gwt.output.concentration().get_data(totim=map_totim)
    lakec = gwt.lak.output.concentration().get_data(totim=map_totim).flatten()
    il, jl = np.where(lakibd > 0)
    for i, j in zip(il, jl):
        conc[0, i, j] = lakec[lakibd[i, j] - 1]
    cdisp = np.ma.masked_where((conc >= 1.0e29) | (conc < MASK_BELOW), conc)
    vmax = max(float(conc[conc < 1.0e29].max()), MASK_BELOW * 10.0)
    norm = LogNorm(vmin=MASK_BELOW, vmax=vmax)
    with styles.USGSMap():
        fig, axs = plt.subplots(2, 2, figsize=(7, 9), dpi=150, constrained_layout=True)
        for ip, ilay in enumerate(map_layers):
            ax = axs.flatten()[ip]
            pmv = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=ilay)
            pmv.plot_inactive(color_noflow="0.8")
            if ilay == 0:
                pmv.plot_array(lake_overlay, cmap=lake_cmap, vmin=0, vmax=1, alpha=0.6)
            ca = pmv.plot_array(cdisp, norm=norm)
            pmv.plot_bc("CHD-1", color="royalblue")
            if s in MCL:
                cs = pmv.contour_array(
                    conc,
                    masked_values=[1.0e30],
                    colors="white",
                    linewidths=1.5,
                    levels=[MCL[s]],
                )
                ax.clabel(cs, fmt=f"{MCL[s]:g}", fontsize=7, colors="white")
            if ip == 0:
                ax.plot(
                    xc,
                    yc,
                    marker="*",
                    ms=11,
                    mfc="magenta",
                    mec="k",
                    mew=0.6,
                    zorder=6,
                    clip_on=False,
                )
                ax.annotate(
                    "(1, 18, 18)",
                    xy=(xc, yc),
                    xytext=(xc - 3400.0, yc + 2200.0),
                    textcoords="data",
                    ha="right",
                    va="center",
                    fontsize=7,
                    color="black",
                    zorder=7,
                    arrowprops=dict(
                        arrowstyle="-", color="black", lw=0.6, shrinkA=0, shrinkB=4
                    ),
                )
            styles.heading(
                ax=ax, letter=chr(ord("A") + ip), heading=f"Model layer {ilay + 1}"
            )
            ax.set_aspect("equal")
            ax.set_xlabel("x position (ft)")
            ax.set_ylabel("y position (ft)")
            ticks = [t for t in decades if MASK_BELOW * 0.999 <= t <= vmax * 1.001]
            cb = fig.colorbar(ca, ax=ax, shrink=0.5, ticks=ticks)
            cb.ax.yaxis.set_major_formatter(plain)
            cb.ax.minorticks_off()
        mcl_note = f", white = MCL {MCL[s]:g} mg/L" if s in MCL else ""
        fig.suptitle(
            f"{labels[s]} at {map_year:.0f} years (blue = constant head{mcl_note})",
            fontsize=11,
            fontweight="bold",
        )
        fig.savefig(fig_path / f"nitrate-nitrite-conc-map-{s}.png", dpi=150)

### Breakthrough and mass in storage

Two panels: (A) the concentration of each species over time at the observation cell (log scale, so the smaller nitrite and dinitrogen curves are visible), and (B) the total dissolved mass of each species in the aquifer. Nitrate arrives first; nitrite and dinitrogen build up behind it as the reaction proceeds, and all three respond after year 10 when the lake source is reduced.

In [ ]:
colors = {"no3": "tab:blue", "no2": "tab:red", "n2": "tab:green"}
times, mass = total_mass_in_storage(sim)

with styles.USGSPlot():
    fig, (axa, axb) = plt.subplots(
        2, 1, sharex=True, figsize=(7, 4.5), dpi=150, constrained_layout=True
    )
    ymax = 0.0
    for s in SPECIES:
        cobj = sim.get_model(s).output.concentration()
        t = np.array(cobj.times) / 365.0
        series = cobj.get_alldata()[:, obs[0], obs[1], obs[2]]
        axa.plot(t, series, color=colors[s], label=labels[s])
        ymax = max(ymax, float(np.max(series)))
    axa.set_yscale("log")
    axa.set_ylim(0.01, 1.5 * ymax)
    axa.yaxis.set_major_formatter(plain)
    axa.set_ylabel("concentration (mg/L)")
    styles.heading(ax=axa, letter="A", heading="Concentration at cell (1, 18, 18)")
    for s in SPECIES:
        axb.plot(times, mass[s], color=colors[s], label=labels[s])
    styles.graph_legend(ax=axb)
    axb.set_ylim(bottom=0.0)
    axb.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    axb.set_ylabel("Total mass in storage (kg)")
    styles.heading(ax=axb, letter="B", heading="Mass in aquifer storage")
    axb.set_xlim(0.0, 25.0)
    axb.set_xlabel("time (years)")
    fig.align_ylabels((axa, axb))
    fig.savefig(fig_path / "nitrate-nitrite-timeseries.png", dpi=150)

**Recap.** This example used the API for something MODFLOW 6 will not do on its own: couple three transport models through a first-order reaction chain. The `src_rhs` callback wrote reaction mass rates into a SRC package so the terms are booked in every budget, while `operator_split` overwrote the concentration state directly — the same lifecycle and the same `get_value` / `get_value_ptr` memory access from the earlier notebooks, now used to inject chemistry between transport time steps. Switch `reaction_method` to `"operator_split"` and re-run from the build cell to see the reaction as a budget discrepancy instead.